In [ ]:
!pip install -q -U langchain langchain-openai langchain-core langgraph python-dotenv

# Agentes con `create_agent` en LangChain

Este notebook muestra, de forma didáctica, cómo se construye el agente que usa `AgentService` en `02_agent_api` (ver `src/services/agent_service.py`), apoyándose en `langchain.agents.create_agent`.

No nos enfocamos en el historial de conversación ni en funciones auxiliares del servicio; el objetivo es entender:

- Qué es `create_agent` y qué parámetros acepta.
- Cómo una **tool** le agrega capacidades al modelo (ej. consultar el clima).
- Cómo agregar **más tools** para extender el agente con nuevas funcionalidades.

In [13]:
import getpass
import os

from dotenv import load_dotenv

# Carga el .env que está a la misma altura que este notebook (02_agent_api/notebooks/.env)
load_dotenv(".env")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OPENAI_API_KEY: ")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")  # cambia el modelo si lo necesitas

## ¿Qué hace `create_agent`?

`create_agent` (documentación: https://reference.langchain.com/python/langchain/agents/factory/create_agent) construye un grafo (LangGraph) que:

1. Llama al modelo con la lista de mensajes (agregando el `system_prompt` al inicio).
2. Si la respuesta del modelo incluye `tool_calls`, ejecuta esas tools y agrega el resultado como `ToolMessage`.
3. Vuelve a llamar al modelo con el resultado de la tool.
4. Repite el ciclo hasta que el modelo responda sin solicitar más tools.

Parámetros principales usados aquí:

- `model`: el LLM a usar (string tipo `"openai:gpt-4o-mini"` o una instancia como `ChatOpenAI`).
- `tools`: lista de funciones decoradas con `@tool` (o instancias de `BaseTool`).
- `system_prompt`: instrucciones fijas que se anteponen a la conversación.

## 1. Una tool básica: clima

Así como en `src/services/tools.py` del API, definimos una tool decorada con `@tool`. El *docstring* es importante: LangChain lo usa como la descripción que el modelo ve para decidir cuándo llamar la tool.

In [14]:
from langchain_core.tools import tool


@tool
def get_weather(location: str) -> str:
    """Devuelve el pronóstico del clima actual para la ubicación indicada."""
    return f"El clima en {location} es soleado con una temperatura de 22°C."

## 2. Creando el agente con `create_agent`

Igual que en `AgentService.__init__`, instanciamos el LLM y creamos el agente pasando la(s) tool(s) disponibles.

In [15]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=OPENAI_MODEL)

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="Eres un asistente útil y amigable. Responde siempre en español.",
)

ImportError: cannot import name 'ToolCallTransformer' from 'langgraph.prebuilt' (unknown location)

## 3. Invocando el agente

El agente recibe un diccionario con la clave `"messages"`. Le hacemos una pregunta que requiere usar la tool de clima.

In [ ]:
from langchain_core.messages import HumanMessage

response = agent.invoke({"messages": [HumanMessage(content="¿Qué clima hay en Bogotá?")]})

for message in response["messages"]:
    message.pretty_print()

Nota cómo la lista de mensajes incluye: el `HumanMessage` original, el `AIMessage` con el `tool_call`, el `ToolMessage` con el resultado de `get_weather`, y finalmente el `AIMessage` con la respuesta final.

## 4. Modo streaming

También podemos ver paso a paso cómo el grafo va actualizando el estado con `stream_mode="updates"`.

In [ ]:
for chunk in agent.stream(
    {"messages": [HumanMessage(content="¿Cómo está el clima en Medellín?")]},
    stream_mode="updates",
):
    print(chunk)

## 5. Agregando más tools

La gran ventaja de `create_agent` es que agregar funcionalidades nuevas es tan simple como definir otra función con `@tool` y añadirla a la lista `tools`. Aquí agregamos tres tools de demostración adicionales:

- `get_current_time`: sin argumentos.
- `calculate`: recibe una expresión matemática.
- `convert_currency`: recibe varios argumentos tipados.

In [ ]:
from datetime import datetime


@tool
def get_current_time() -> str:
    """Devuelve la fecha y hora actual."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@tool
def calculate(expression: str) -> str:
    """Evalúa una expresión matemática simple, por ejemplo '2 + 2 * 3'."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as exc:
        return f"No pude calcular la expresión: {exc}"


@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convierte un monto entre dos monedas usando tasas de cambio fijas de demostración."""
    demo_rates = {
        "USD_COP": 4000,
        "COP_USD": 1 / 4000,
        "USD_EUR": 0.92,
        "EUR_USD": 1 / 0.92,
    }
    key = f"{from_currency.upper()}_{to_currency.upper()}"
    rate = demo_rates.get(key)
    if rate is None:
        return f"No tengo una tasa de cambio de demostración para {from_currency} -> {to_currency}."
    converted = amount * rate
    return f"{amount} {from_currency.upper()} equivalen a {converted:.2f} {to_currency.upper()} (tasa de demostración)."

## 6. Un agente con múltiples tools

Creamos un nuevo agente pasando las cuatro tools. El modelo decidirá cuáles usar según la pregunta.

In [ ]:
multi_tool_agent = create_agent(
    model=llm,
    tools=[get_weather, get_current_time, calculate, convert_currency],
    system_prompt=(
        "Eres un asistente útil y amigable. Responde siempre en español. "
        "Usa las herramientas disponibles cuando sea necesario para responder con precisión."
    ),
)

Le hacemos una pregunta que requiere combinar varias tools en una sola respuesta.

In [ ]:
response = multi_tool_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="¿Qué hora es, cómo está el clima en Cali y cuánto son 50 USD en COP?"
            )
        ]
    }
)

for message in response["messages"]:
    message.pretty_print()

## Conclusión

- `create_agent(model, tools, system_prompt)` es todo lo que se necesita para tener un agente con ciclo de tool-calling.
- Cada nueva funcionalidad se agrega como una función decorada con `@tool`, con un docstring claro (es lo que el modelo usa para decidir cuándo invocarla).
- El agente decide automáticamente qué tool(s) usar y puede combinar varias en una sola respuesta.
- Este es el mismo patrón que usa `AgentService` en `src/services/agent_service.py` del API, solo que allí además se persiste el historial de conversación y se calculan tokens/tiempos, aspectos que quedaron fuera de este notebook a propósito.